In [1]:
#!pip install tensorflow_addons
#!pip install patchify
#!pip install positional_encodings
import cv2,scipy
import random
from scipy import ndimage
import natsort
from natsort import natsorted
from patchify import patchify, unpatchify
import numpy as np
from keras.applications import imagenet_utils
import tensorflow_addons as tfa
from keras.engine import training
from keras.layers import VersionAwareLayers
from keras.utils import data_utils
from keras.utils import layer_utils
import pickle
from skimage.filters import gabor_kernel
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
import tensorflow as tf
from tensorflow.keras import regularizers
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from heapq import nsmallest,nlargest
from scipy.linalg import svd
from scipy.sparse.linalg import LinearOperator
from positional_encodings.tf_encodings import TFPositionalEncoding3D, TFSummer
from scipy.linalg import lstsq
from sklearn.preprocessing import MinMaxScaler
from scipy.signal.signaltools import wiener
import os
from keras import layers
from keras import backend
from sklearn.utils import shuffle
import pandas as pd
from skimage.segmentation import chan_vese
from keras.models import load_model
import tensorflow as tf
from tensorflow.keras.utils import to_categorical,plot_model
from sklearn.preprocessing import MinMaxScaler, StandardScaler
scaler = MinMaxScaler()
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import keras
from tqdm import tqdm
from tensorflow.keras.layers import Dense,Add,Conv3D,MaxPool3D,GlobalAveragePooling3D,GlobalMaxPooling3D,Embedding, Activation,Add,Multiply, Dot, Softmax, Lambda, Conv2D, Conv2DTranspose,Activation, BatchNormalization,TimeDistributed,AveragePooling2D,Flatten,Dense,Add,ReLU, LSTM
from tensorflow.keras.layers import ZeroPadding2D,UpSampling2D, MaxPool3D,Input, Concatenate,Multiply,LayerNormalization, Lambda, SeparableConv2D,MaxPooling2D,GlobalMaxPooling2D,GlobalAveragePooling2D,Dropout
from keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import ResNet50, ResNet101,EfficientNetB7,Xception,VGG19,MobileNetV2,ResNet101V2
from tensorflow.keras.applications import EfficientNetB7
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import Recall, Precision, Accuracy
from tensorflow.keras import backend as K
from PIL import Image
from tensorflow.python.util.tf_export import keras_export

C:\Users\mahesh.inamdar\AppData\Local\Programs\Python\Python39\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
C:\Users\mahesh.inamdar\AppData\Local\Programs\Python\Python39\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.10.0 and is not supported. 
Some things might work, some things might not.
If yo

In [4]:
def model(input_1):
    Efficient=ResNet50(include_top=False,weights="imagenet",input_tensor=input_1,input_shape=None,pooling=None,classes=1000)
    Layer1=GlobalAveragePooling2D()(Efficient.get_layer('conv5_block3_out').output)
    return Model(input_1,Layer1)

inputs1=tf.keras.Input(shape=(512,512,3))
model=model(inputs1)

In [13]:
dir_train='C:/Users/mahesh.inamdar/Desktop/codes/Sets/Set 9010/train/'
dir_test='C:/Users/mahesh.inamdar/Desktop/codes/Sets/Set 9010/test/'

import os
import cv2
from tqdm import tqdm
import numpy as np
from scipy import ndimage
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils import shuffle
from xgboost import XGBClassifier
import pickle
from skimage.filters import threshold_otsu
from skimage.segmentation import clear_border
from skimage.morphology import closing, square
from scipy import ndimage
import matplotlib.patches as mpatches
from skimage.color import label2rgb
from skimage.measure import label, regionprops,find_contours
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
from skimage.draw import polygon_perimeter
from math import atan2, pi, atan
from lazypredict.Supervised import LazyClassifier

def preprocessing(im1):
    im1[im1==255]=0
    im2 = cv2.medianBlur(im1,51)
    th,img2=cv2.threshold(im2, 10, 255, cv2.THRESH_BINARY)
    img2[img2==255]=1
    image=np.multiply(img2,im1)
    thresh = threshold_otsu(image)
    bw = closing(image > thresh, square(3))
    cleared = clear_border(bw)
    label_image = label(cleared)
    image_label_overlay = label2rgb(label_image, image=image, bg_label=0)
    contours = find_contours(image, 0.8)
    bounding_boxes = []
    
    for contour in contours:
        Xmin = np.min(contour[:,0])
        Xmax = np.max(contour[:,0])
        Ymin = np.min(contour[:,1])
        Ymax = np.max(contour[:,1])

        bounding_boxes.append([Xmin, Xmax, Ymin, Ymax])

    with_boxes  = np.copy(image)

    for box in bounding_boxes:
        #[Xmin, Xmax, Ymin, Ymax]
        r = [box[0],box[1],box[1],box[0], box[0]]
        c = [box[3],box[3],box[2],box[2], box[3]]
        rr, cc = polygon_perimeter(r, c, with_boxes.shape)
        with_boxes[rr, cc] = 5 #set color white
        
    #fig, ax = plt.subplots(figsize=(10, 6))
    poly=[]
    
    for region in regionprops(label_image):
        # take regions with large enough areas
        if region.area >= 10000:
                # draw rectangle around segmented coins
            minr, minc, maxr, maxc = region.bbox
            rect = mpatches.Rectangle((minc, minr), maxc - minc, maxr - minr,
                                          fill=False, edgecolor='red', linewidth=2)
            poly.append(region.bbox)
            #ax.add_patch(rect)

    try:
        a,b,c,d=poly[-1]
        return cv2.resize(image[a:c,b:d], (512,512), interpolation= cv2.INTER_LINEAR)
    except IndexError:
        return cv2.resize(image, (512,512), interpolation= cv2.INTER_LINEAR)

def get_label(dir):
    if 'NS' in dir:
        return 0
    elif 'AS' in dir:
        return 1
    elif 'CS' in dir:
        return 2
    
def resize_image(image, target_size=(512, 512)):
    resized_image = cv2.resize(image, target_size)
    return resized_image

def extract_patches(image, patch_size):
    patches = []
    height, width = image.shape[:2]
    for y in range(0, height - patch_size[0] + 1, patch_size[0]):
        for x in range(0, width - patch_size[1] + 1, patch_size[1]):
            patch = image[y:y+patch_size[0], x:x+patch_size[1]]
            patches.append(patch)
    return np.array(patches)

def wld_descriptor_mod(image, num_neighbors=8, num_orientations=8,num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3):
    """
    Compute the Weber Local Descriptor (WLD) for the given image.
    
    Args:
        image (numpy.ndarray): Input image.
        num_neighbors (int): Number of neighbors to consider for differential excitation.
        num_orientations (int): Number of dominant orientations to quantize the gradients.
        
    Returns:
        numpy.ndarray: WLD descriptor histogram.
    """
    # Compute differential excitation
    diff_excitation = differential_excitation(image, num_neighbors)
    segment_min = diff_excitation.min()
    segment_max = diff_excitation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    diff_excitation_hist, _ = np.histogram(diff_excitation, bins=segment_bins, density=True)
    
    # Compute orientation
    orientation = structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0)
    segment_min = orientation.min()
    segment_max = orientation.max()
    segment_bins = np.linspace(segment_min, segment_max, endpoint=True)
    orientation_hist, _ = np.histogram(orientation, bins=segment_bins, density=True)
    
    diff_excitation = np.nan_to_num(diff_excitation, nan=0.0)

    # Set default value for num_diff_exc_bins if None
    num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    # Combine differential excitation and orientation into the WLD descriptor histogram
    if num_diff_exc_bins is None:
        num_diff_exc_bins = int(np.max(diff_excitation) - np.min(diff_excitation)) + 1
    num_orientations=8
    wld_2d_hist = np.zeros((num_diff_exc_bins, num_orientations))
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            j = int(diff_excitation[y, x])
            t = int(orientation[y, x])
            wld_2d_hist[j, t] += 1

    # Encode the 2D histogram into the 1D histogram H
    wld_1d_hist = encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment)

    return np.array(np.ravel(wld_2d_hist))

def encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment):
    """
    Encode the 2D WLD histogram into a 1D histogram.

    Args:
        wld_2d_hist (numpy.ndarray): 2D WLD histogram.
        num_segments (int): Number of segments to divide the differential excitation range.
        num_bins_per_segment (int): Number of bins within each differential excitation segment.

    Returns:
        numpy.ndarray: 1D WLD descriptor histogram.
    """
    num_orientations = wld_2d_hist.shape[1]
    diff_exc_range = np.pi
    segment_width = diff_exc_range / num_segments

    # Divide each column (orientation) into segments and concatenate
    wld_1d_hist = []
    for t in range(num_orientations):
        column_hist = wld_2d_hist[:, t]
        segments = np.array_split(column_hist, num_segments)
        segment_min = np.min(np.ravel(wld_2d_hist))
        segment_max = np.max(np.ravel(wld_2d_hist))
        for segment in segments:
            if segment.size > 0:  # Check if segment is not empty
                segment_bins = np.linspace(segment_min, segment_max, num_bins_per_segment + 1, endpoint=True)
                segment_hist, _ = np.histogram(segment, bins=segment_bins, density=True)
                segment_hist[np.isnan(segment_hist)] = 0  # Replace NaN values with 0
                segment_hist[np.isinf(segment_hist)] = 0
                wld_1d_hist.extend(segment_hist)
            else:
                # If segment is empty, assign default values to the corresponding bins
                wld_1d_hist.extend([0.0] * num_bins_per_segment)

    return np.array(wld_1d_hist)

def differential_excitation(image, num_neighbors):
    """
    Compute the differential excitation component of the WLD descriptor.
    
    Args:
        image (numpy.ndarray): Input image.
        num_neighbors (int): Number of neighbors to consider.
        
    Returns:
        numpy.ndarray: Differential excitation values.
    """
    diff_excitation = np.zeros_like(image, dtype=np.float64)
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            neighbors = []
            for dy in [-1, 0, 1]:
                for dx in [-1, 0, 1]:
                    if dy == dx == 0:
                        continue
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < image.shape[0] and 0 <= nx < image.shape[1]:
                        neighbors.append(image[ny, nx])
            neighbors = np.array(neighbors)
            diff_excitation[y, x] = np.arctan(np.sum(neighbors - image[y, x]) / (image[y, x]+1e-6))
    return diff_excitation

def structure_tensor_orientation(image, num_orientations=8,window_size=3, sigma=1.0):
    """
    Compute the orientation component of the WLD descriptor using the structure tensor approach.

    Args:
        image (numpy.ndarray): Input image.
        window_size (int): Size of the local neighborhood window for computing the structure tensor.
        sigma (float): Standard deviation for Gaussian kernel used for smoothing.

    Returns:
        numpy.ndarray: Orientations computed using the structure tensor.
    """
    # Compute gradients
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    # Compute structure tensor components
    ix2 = sobel_x ** 2
    iy2 = sobel_y ** 2
    ixy = sobel_x * sobel_y

    # Smooth the structure tensor components
    kernel = cv2.getGaussianKernel(window_size, sigma)
    ix2 = cv2.sepFilter2D(ix2, -1, kernel, kernel)
    iy2 = cv2.sepFilter2D(iy2, -1, kernel, kernel)
    ixy = cv2.sepFilter2D(ixy, -1, kernel, kernel)

    # Compute eigenvalues and orientations
    orientation = np.zeros_like(image, dtype=np.float64)
    num_orientations=8
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            tensor = np.array([[ix2[y, x], ixy[y, x]], [ixy[y, x], iy2[y, x]]])
            eigenvalues, eigenvectors = np.linalg.eig(tensor)
            orientation[y, x] = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
    
    if num_orientations is not None:
        orientation = (orientation % (2 * pi)) * num_orientations / (2 * pi)
        return orientation.astype(int)
    else:
        return orientation.astype(int)

def wld_descriptor_multiscale(image, num_orientations=8, num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3, num_scales=4):
    """
    Compute the Weber Local Descriptor (WLD) for the given image at multiple scales.

    Args:
        image (numpy.ndarray): Input image.
        num_orientations (int): Number of dominant orientations to quantize.
        num_diff_exc_bins (int, optional): Number of bins for differential excitation values.
            If None, it is set to the maximum intensity value in the image.
        num_segments (int): Number of segments to divide the differential excitation range.
        num_bins_per_segment (int): Number of bins within each differential excitation segment.
        num_scales (int): Number of scales to compute the WLD descriptor.

    Returns:
        numpy.ndarray: 1D concatenated multi-scale WLD descriptor histogram.
    """
    # Create a Gaussian pyramid for the input image
    pyramid = [image.astype(np.float32)]  # Add the original image as the first level
    for _ in range(num_scales - 1):
        pyramid.append(cv2.pyrDown(pyramid[-1]))

    # Compute WLD descriptor at each scale and concatenate
    wld_descriptors = []
    for img in pyramid:
        #print('check',num_orientations)
        wld_descriptors.append(wld_descriptor_mod(image, num_orientations, num_diff_exc_bins, num_segments, num_bins_per_segment))

    wld_multiscale = np.concatenate(wld_descriptors)
    return wld_multiscale

# Load and preprocess the images
def load_images(directory,train,sig):
    if train:
        print('-----TRAIN-----')
        labels = []
        Xtr=np.zeros(shape=(4367,2305))
        ntr=0
        for filename in tqdm(os.listdir(directory)):
            img = preprocessing(resize_image(cv2.imread(os.path.join(directory,filename), cv2.IMREAD_GRAYSCALE)))
            labels.append(get_label(filename))
            patches=extract_patches(img,(256,256))
            resnetpred=model.predict(np.expand_dims(cv2.merge((img,img,img)),0))
            con=np.concatenate((wld_descriptor_multiscale(patches[0]),wld_descriptor_multiscale(patches[1]),
                                    wld_descriptor_multiscale(patches[2]),wld_descriptor_multiscale(patches[3])))
            Xtr[ntr,0:np.shape(resnetpred)[1]]=resnetpred
            Xtr[ntr,np.shape(resnetpred)[1]:np.shape(resnetpred)[1]+len(con)]= con
            Xtr[ntr,-1]=get_label(filename)
            ntr+=1
        return np.array(Xtr), np.array(labels)
    else:
        print('-----TEST-----')
        Xte=np.zeros(shape=(483,2305))
        nte=0
        labels = []
        for filename in tqdm(os.listdir(directory)):
            img = preprocessing(resize_image(cv2.imread(os.path.join(directory,filename), cv2.IMREAD_GRAYSCALE)))
            labels.append(get_label(filename))
            patches=extract_patches(img,(256,256))
            resnetpred=model.predict(np.expand_dims(cv2.merge((img,img,img)),0))
            con=np.concatenate((wld_descriptor_multiscale(patches[0]),wld_descriptor_multiscale(patches[1]),
                                    wld_descriptor_multiscale(patches[2]),wld_descriptor_multiscale(patches[3])))
            Xte[nte,0:np.shape(resnetpred)[1]]=resnetpred
            Xte[nte,np.shape(resnetpred)[1]:np.shape(resnetpred)[1]+len(con)]= con
            Xte[nte,-1]=get_label(filename)
            nte+=1
        return np.array(Xte), np.array(labels)

X_train,y_train=load_images(dir_train,True,False)
X_test,y_test=load_images(dir_test,False,False)

print(np.shape(X_train),np.shape(y_train),np.shape(X_test),np.shape(y_test))
np.savetxt('C:/Users/mahesh.inamdar/Desktop/codes/Sets/Set 9010/feature_histW2ResnetModWLDSet9010_train.csv', X_train, delimiter=',')
np.savetxt('C:/Users/mahesh.inamdar/Desktop/codes/Sets/Set 9010/feature_histW2ResnetModWLDSet9010_test.csv', X_test, delimiter=',')
print('Data Stored in csv')



-----TRAIN-----


  0%|                                                                                         | 0/4367 [00:00<?, ?it/s]

1/1 [==============================] - 0s 31ms/step


  0%|                                                                              | 1/4367 [00:41<50:52:19, 41.95s/it]

1/1 [==============================] - 0s 31ms/step


...

1/1 [==============================] - 0s 32ms/step


100%|████████████████████████████████████████████████████████████████████████████▊| 482/483 [22:14:10<00:41, 41.96s/it]

1/1 [==============================] - 0s 41ms/step


100%|████████████████████████████████████████████████████████████████████████████| 483/483 [22:14:52<00:00, 165.82s/it]


(4367, 2305) (4367,) (483, 2305) (483,)
Data Stored in csv


In [11]:

import os
import cv2
from tqdm import tqdm
import numpy as np
from scipy import ndimage
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils import shuffle
from xgboost import XGBClassifier
import pickle
from skimage.filters import threshold_otsu
from skimage.segmentation import clear_border
from skimage.morphology import closing, square
from scipy import ndimage
import matplotlib.patches as mpatches
from skimage.color import label2rgb
from skimage.measure import label, regionprops,find_contours
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
from skimage.draw import polygon_perimeter
from math import pi, atan2, atan
from lazypredict.Supervised import LazyClassifier

# Define the directories
dir_normal = 'C:\\Users\\yaswa\\learn\\Neural_networks\\research_ws\\Brain_ct_scan\\mal_dataset_thick\\mal_normal'
dir_thin = 'C:\\Users\\yaswa\\learn\\Neural_networks\\research_ws\\Brain_ct_scan\\mal_dataset_thick\\mal_acute'
dir_chronic = 'C:\\Users\\yaswa\\learn\\Neural_networks\\research_ws\\Brain_ct_scan\\mal_dataset_thick\\mal_chronic'

dir_train='C:/Users/mahesh.inamdar/Desktop/codes/classes/run/train'
dir_test='C:/Users/mahesh.inamdar/Desktop/codes/classes/run/test'

def preprocessing(im1):
    im1[im1==255]=0
    im2 = cv2.medianBlur(im1,51)
    th,img2=cv2.threshold(im2, 10, 255, cv2.THRESH_BINARY)
    img2[img2==255]=1
    image=np.multiply(img2,im1)
    thresh = threshold_otsu(image)
    bw = closing(image > thresh, square(3))
    cleared = clear_border(bw)
    label_image = label(cleared)
    image_label_overlay = label2rgb(label_image, image=image, bg_label=0)
    contours = find_contours(image, 0.8)
    bounding_boxes = []
    
    for contour in contours:
        Xmin = np.min(contour[:,0])
        Xmax = np.max(contour[:,0])
        Ymin = np.min(contour[:,1])
        Ymax = np.max(contour[:,1])

        bounding_boxes.append([Xmin, Xmax, Ymin, Ymax])

    with_boxes  = np.copy(image)

    for box in bounding_boxes:
        #[Xmin, Xmax, Ymin, Ymax]
        r = [box[0],box[1],box[1],box[0], box[0]]
        c = [box[3],box[3],box[2],box[2], box[3]]
        rr, cc = polygon_perimeter(r, c, with_boxes.shape)
        with_boxes[rr, cc] = 5 #set color white
        
    #fig, ax = plt.subplots(figsize=(10, 6))
    poly=[]
    
    for region in regionprops(label_image):
        # take regions with large enough areas
        if region.area >= 10000:
                # draw rectangle around segmented coins
            minr, minc, maxr, maxc = region.bbox
            rect = mpatches.Rectangle((minc, minr), maxc - minc, maxr - minr,
                                          fill=False, edgecolor='red', linewidth=2)
            poly.append(region.bbox)
            #ax.add_patch(rect)

    try:
        a,b,c,d=poly[-1]
        return cv2.resize(image[a:c,b:d], (512,512), interpolation= cv2.INTER_LINEAR)
    except IndexError:
        return cv2.resize(image, (512,512), interpolation= cv2.INTER_LINEAR)


def ltp_descriptor(image, radius=3, neighbors=8, threshold=5):
    """
    Compute the Local Ternary Pattern (LTP) descriptor.

    Args:
        image (numpy.ndarray): Input image.
        radius (int): Radius of the neighborhood.
        neighbors (int): Number of neighbors to consider.
        threshold (int): Threshold value for ternary coding.

    Returns:
        numpy.ndarray: LTP descriptor histogram.
    """
    ltp_hist = np.zeros(3 ** neighbors, dtype=np.uint8)
    for y in range(radius, image.shape[0] - radius):
        for x in range(radius, image.shape[1] - radius):
            center = image[y, x]
            code = 0
            for n in range(neighbors):
                x_offset = x + radius * np.cos(2 * pi * n / neighbors)
                y_offset = y - radius * np.sin(2 * pi * n / neighbors)
                neighbor = image[int(y_offset), int(x_offset)]
                code = code * 3 + (neighbor >= center + threshold) + (neighbor <= center - threshold)
            ltp_hist[code] += 1
    return ltp_hist

def lbp_descriptor(image, radius=3, neighbors=8):
    """
    Compute the Local Binary Pattern (LBP) descriptor.

    Args:
        image (numpy.ndarray): Input image.
        radius (int): Radius of the neighborhood.
        neighbors (int): Number of neighbors to consider.

    Returns:
        numpy.ndarray: LBP descriptor histogram.
    """
    lbp_hist = np.zeros(2 ** neighbors, dtype=np.uint8)
    for y in range(radius, image.shape[0] - radius):
        for x in range(radius, image.shape[1] - radius):
            center = image[y, x]
            code = 0
            for n in range(neighbors):
                x_offset = x + radius * np.cos(2 * pi * n / neighbors)
                y_offset = y - radius * np.sin(2 * pi * n / neighbors)
                neighbor = image[int(y_offset), int(x_offset)]
                code = code * 2 + (neighbor >= center)
            lbp_hist[code] += 1
    return lbp_hist


def wld_descriptor(image, num_orientations=8, num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3):
    """
    Compute the Weber Local Descriptor (WLD) for the given image.
    
    Args:
        image (numpy.ndarray): Input image.
        num_neighbors (int): Number of neighbors to consider for differential excitation.
        num_orientations (int): Number of dominant orientations to quantize the gradients.
        
    Returns:
        numpy.ndarray: WLD descriptor histogram.
    """
    # Compute differential excitation
    diff_excitation = differential_excitation(image)

    # Compute orientation
    orientation = gradient_orientation(image, num_orientations)

    # Replace NaN values in diff_excitation with a suitable value (e.g., 0)
    diff_excitation = np.nan_to_num(diff_excitation, nan=0.0)

    # Set default value for num_diff_exc_bins if None
    num_diff_exc_bins = num_diff_exc_bins or int(np.max(diff_excitation) - np.min(diff_excitation)) + 1

    wld_2d_hist = np.zeros((num_diff_exc_bins, num_orientations))
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            j = int(diff_excitation[y, x])
            t = int(orientation[y, x])
            wld_2d_hist[j, t] += 1

    # Encode the 2D histogram into the 1D histogram H
    wld_1d_hist = encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment)

    return np.ravel(np.array(wld_2d_hist))

def encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment):
    """
    Encode the 2D WLD histogram into a 1D histogram.

    Args:
        wld_2d_hist (numpy.ndarray): 2D WLD histogram.
        num_segments (int): Number of segments to divide the differential excitation range.
        num_bins_per_segment (int): Number of bins within each differential excitation segment.

    Returns:
        numpy.ndarray: 1D WLD descriptor histogram.
    """
    num_orientations = wld_2d_hist.shape[1]
    diff_exc_range = np.pi
    segment_width = diff_exc_range / num_segments

    # Divide each column (orientation) into segments and concatenate
    wld_1d_hist = []
    for t in range(num_orientations):
        column_hist = wld_2d_hist[:, t]
        segments = np.array_split(column_hist, num_segments)
        for segment in segments:
            if segment.size > 0:  # Check if segment is not empty
                segment_min = segment.min()
                segment_max = segment.max()
                segment_bins = np.linspace(segment_min, segment_max, num_bins_per_segment + 1, endpoint=True)
                segment_hist, _ = np.histogram(segment, bins=segment_bins, density=True)
                wld_1d_hist.extend(segment_hist)
            else:
                # If segment is empty, assign default values to the corresponding bins
                wld_1d_hist.extend([0.0] * num_bins_per_segment)

    return np.array(wld_1d_hist)

def differential_excitation(image):
    height, width = image.shape
    # Initialize an empty array to store the differential excitation values
    diff_excitation = np.zeros((height, width), dtype=np.uint8)

    # Iterate over each pixel in the image
    for y in range(1, height - 1):
        for x in range(1, width - 1):
            # Compute the difference between the central pixel and its neighbors
            central_pixel = image[y, x]
            neighbors = [
                image[y - 1, x - 1], image[y - 1, x], image[y - 1, x + 1],
                image[y, x - 1], central_pixel, image[y, x + 1],
                image[y + 1, x - 1], image[y + 1, x], image[y + 1, x + 1]
            ]
            differences = [abs(central_pixel - neighbor) for neighbor in neighbors]
            # Take the mean of the differences and assign it to the central pixel position
            diff_excitation[y, x] = np.mean(differences)

    return diff_excitation

def gradient_orientation(image, num_orientations):
    """
    Compute the gradient orientation component of the WLD descriptor using the Sobel operator.

    Args:
        image (numpy.ndarray): Input image.
        num_orientations (int): Number of dominant orientations to quantize.

    Returns:
        numpy.ndarray: Gradient orientations.
    """
    # Compute gradients using the Sobel operator
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    # Compute gradient orientation
    orientation = np.zeros_like(image, dtype=np.float64)
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            dx = sobel_x[y, x]
            dy = sobel_y[y, x]
            orientation[y, x] = (atan2(dy, dx) + pi) % (2 * pi) * num_orientations / (2 * pi)

    return orientation.astype(int)

    
def resize_image(image, target_size=(512, 512)):
    resized_image = cv2.resize(image, target_size)
    return resized_image

def fused_descriptor(image, num_orientations=8, num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3, ltp_radius=3, num_scales=5,ltp_neighbors=8, ltp_threshold=5, lbp_radius=3, lbp_neighbors=8):
    """
    Compute a fused descriptor combining WLD, LTP, and LBP.

    Args:
        image (numpy.ndarray): Input image.
        num_orientations (int): Number of dominant orientations for WLD.
        num_diff_exc_bins (int, optional): Number of bins for differential excitation values in WLD.
        num_segments (int): Number of segments for differential excitation in WLD.
        num_bins_per_segment (int): Number of bins within each differential excitation segment in WLD.
        ltp_radius (int): Radius of the neighborhood for LTP.
        ltp_neighbors (int): Number of neighbors to consider for LTP.
        ltp_threshold (int): Threshold value for ternary coding in LTP.
        lbp_radius (int): Radius of the neighborhood for LBP.
        lbp_neighbors (int): Number of neighbors to consider for LBP.

    Returns:
        numpy.ndarray: Fused descriptor combining WLD, LTP, and LBP.
    """
    # Compute WLD descriptor
    wld_descriptor_hist = wld_descriptor(image, num_orientations, num_diff_exc_bins, num_segments, num_bins_per_segment)

    # Compute LTP descriptor
    ltp_descriptor_hist = ltp_descriptor(image, ltp_radius, ltp_neighbors, ltp_threshold)

    # Compute LBP descriptor
    lbp_descriptor_hist = lbp_descriptor(image, lbp_radius, lbp_neighbors)

    # Concatenate WLD, LTP, and LBP descriptors
    fused_descriptor_hist = np.concatenate((wld_descriptor_hist, ltp_descriptor_hist, lbp_descriptor_hist))

    return fused_descriptor_hist

def get_label(dir):
    if 'NS' in dir:
        return 0
    elif 'AS' in dir:
        return 1
    elif 'CS' in dir:
        return 2

print('Test sampling beings..')
dirpath='C:/Users/mahesh.inamdar/Desktop/codes/classes/run/test'
folders=np.random.choice(os.listdir(dirpath),40)
images=[]
X_test=np.zeros(shape=(100,8633))
y_test=[]
c=0
while c<100:
    n=np.random.choice(folders,1)
    img=np.random.choice(os.listdir(os.path.join(dirpath,n[0])),1)
    w=fused_descriptor(preprocessing(resize_image(cv2.imread(os.path.join(dirpath,n[0],img[0]),0),(512,512))))
    X_test[c,0:len(w)]=w
    y_test.append(get_label(n[0]))
    if c!=100:
        c+=1
    if c>100:
        break
    

print('Train sampling beings..')
dirpath='C:/Users/mahesh.inamdar/Desktop/codes/classes/run/train'
folders=np.random.choice(os.listdir(dirpath),10)
images=[]
X_train=np.zeros(shape=(300,8633))
y_train=[]
a,no,c,N=0,0,0,0
while N<300:
    if N>300:
        break
    if N<300:
        N+=1
    n=np.random.choice(folders,1)
    y_train.append(get_label(n[0]))
    if 'AS' in n[0] and a<100:
        img=np.random.choice(os.listdir(os.path.join(dirpath,n[0])),1)
        w=fused_descriptor(preprocessing(resize_image(cv2.imread(os.path.join(dirpath,n[0],img[0]),0),(512,512))))
        X_train[N,0:len(w)]=w
        a+=1
    elif 'NS' in n[0] and no<100:
        img=np.random.choice(os.listdir(os.path.join(dirpath,n[0])),1)
        w=fused_descriptor(preprocessing(resize_image(cv2.imread(os.path.join(dirpath,n[0],img[0]),0),(512,512))))
        X_train[N,0:len(w)]=w
        no+=1
    elif 'CS' in n[0] and c<100:
        img=np.random.choice(os.listdir(os.path.join(dirpath,n[0])),1)
        w=fused_descriptor(preprocessing(resize_image(cv2.imread(os.path.join(dirpath,n[0],img[0]),0),(512,512))))
        X_train[N,0:len(w)]=w
        c+=1

clf=LazyClassifier(verbose=0,ignore_warnings=True)
models,predictions=clf.fit(X_train,X_test,y_train,y_test)
print(models)

'''
img=preprocessing(resize_image(cv2.imread('C:/Users/mahesh.inamdar/Desktop/codes/classes/run/train/AS_01_thin/Anonymize.Seq3.Ser202.Img145.jpg',0),(512,512)))
W=fused_descriptor(img,num_orientations=8, num_diff_exc_bins=None, num_segments=6, num_bins_per_segment=3)
print(np.shape(W))'''

Test sampling beings..
Train sampling beings..


IndexError: index 300 is out of bounds for axis 0 with size 300

In [15]:
np.shape(y_test)

(100,)

In [16]:
clf=LazyClassifier(verbose=0,ignore_warnings=True)
models,predictions=clf.fit(X_train,X_test,y_train,y_test)
print(models)

 97%|███████████████████████████████████████████████████████████████████████████████▏  | 28/29 [00:40<00:02,  2.03s/it]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016240 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 89907
[LightGBM] [Info] Number of data points in the train set: 300, number of used features: 2194
[LightGBM] [Info] Start training from score -1.139434
[LightGBM] [Info] Start training from score -0.967584
[LightGBM] [Info] Start training from score -1.203973
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No

100%|██████████████████████████████████████████████████████████████████████████████████| 29/29 [00:41<00:00,  1.44s/it]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
AdaBoostClassifier                 0.50               0.53    None      0.51   
NearestCentroid                    0.40               0.51    None      0.36   
Gauss

In [35]:
import numpy as np
import cv2

def differential_excitation(img, kernel_size=5, sigma=1.0):
    """
    Compute differential excitation based on the Gaussian kernel.
    """
    # Compute gradient images
    grad_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=kernel_size)
    grad_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=kernel_size)

    # Compute differential excitation
    diff_excitation = np.sqrt(grad_x ** 2 + grad_y ** 2)

    # Apply Gaussian smoothing
    diff_excitation = cv2.GaussianBlur(diff_excitation, (kernel_size, kernel_size), sigma)

    return diff_excitation

def structure_tensor(img, kernel_size=5, sigma=1.0):
    """
    Compute the structure tensor for orientation estimation.
    """
    # Compute gradient images
    grad_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=kernel_size)
    grad_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=kernel_size)

    # Compute structure tensor components
    grad_xx = grad_x ** 2
    grad_xy = grad_x * grad_y
    grad_yy = grad_y ** 2

    # Apply Gaussian smoothing
    grad_xx = cv2.GaussianBlur(grad_xx, (kernel_size, kernel_size), sigma)
    grad_xy = cv2.GaussianBlur(grad_xy, (kernel_size, kernel_size), sigma)
    grad_yy = cv2.GaussianBlur(grad_yy, (kernel_size, kernel_size), sigma)

    return grad_xx, grad_xy, grad_yy

def wld_orientation(img, kernel_size=5, sigma=1.0):
    """
    Compute orientation for the Weber Local Descriptor (WLD).
    """
    # Compute differential excitation
    diff_exc = differential_excitation(img, kernel_size, sigma)

    # Compute structure tensor components
    grad_xx, grad_xy, grad_yy = structure_tensor(img, kernel_size, sigma)

    # Compute orientation
    numerator = grad_yy - grad_xx
    denominator = 2 * grad_xy
    orientation = np.arctan2(numerator, denominator)

    return orientation, diff_exc

def wld_descriptor(image,kernel_size=5, sigma=1.0):
    num_orientations=8
    num_diff_exc_bins=None
    num_segments=6
    num_bins_per_segment=3
    """
    Compute the Weber Local Descriptor (WLD) for the given image.

    Args:
        image (numpy.ndarray): Input image.
        num_orientations (int): Number of dominant orientations to quantize the gradients.
        num_diff_exc_bins (int, optional): Number of differential excitation bins. If None, it is set to the maximum value of differential excitation plus 1.
        num_segments (int): Number of segments for the 1D histogram encoding.
        num_bins_per_segment (int): Number of bins per segment for the 1D histogram encoding.

    Returns:
        numpy.ndarray: WLD descriptor histogram.
    """

    # Compute differential excitation
    diff_excitation = differential_excitation(image)

    # Compute orientation
    orientation, _ = wld_orientation(image, kernel_size=3, sigma=1.0)

    # Replace NaN values in diff_excitation with a suitable value (e.g., 0)
    diff_excitation = np.nan_to_num(diff_excitation, nan=0.0)

    # Set default value for num_diff_exc_bins if None
    num_diff_exc_bins = num_diff_exc_bins or int(np.max(diff_excitation) - np.min(diff_excitation)) + 1

    wld_2d_hist = np.zeros((num_diff_exc_bins, num_orientations))

    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            j = int(diff_excitation[y, x])
            t = int(orientation[y, x])
            wld_2d_hist[j, t] += 1

    # Encode the 2D histogram into the 1D histogram H
    wld_1d_hist = encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment)

    return wld_1d_hist

def encode_2d_to_1d_histogram(wld_2d_hist, num_segments, num_bins_per_segment):
    """
    Encode the 2D histogram into a 1D histogram with a predefined number of segments and bins per segment.

    Args:
        wld_2d_hist (numpy.ndarray): 2D histogram of differential excitation and orientation.
        num_segments (int): Number of segments for the 1D histogram encoding.
        num_bins_per_segment (int): Number of bins per segment for the 1D histogram encoding.

    Returns:
        numpy.ndarray: 1D histogram encoding of the 2D histogram.
    """
    wld_1d_hist = np.zeros((num_segments, num_bins_per_segment))

    # Iterate over the 2D histogram and populate the 1D histogram
    for j in range(wld_2d_hist.shape[0]):
        for t in range(wld_2d_hist.shape[1]):
            segment_idx = j * num_segments // wld_2d_hist.shape[0]
            bin_idx = t * num_bins_per_segment // wld_2d_hist.shape[1]

            # Check if indices are within valid range
            if segment_idx < num_segments and bin_idx < num_bins_per_segment:
                wld_1d_hist[segment_idx, bin_idx] += wld_2d_hist[j, t]

    return wld_1d_hist.ravel()

def wld_descriptor_multiscale(image):
    num_orientations=8
    num_diff_exc_bins=None
    num_segments=6
    num_bins_per_segment=3
    num_scales=4
    """
    Compute the Weber Local Descriptor (WLD) for the given image at multiple scales.

    Args:
        image (numpy.ndarray): Input image.
        num_orientations (int): Number of dominant orientations to quantize.
        num_diff_exc_bins (int, optional): Number of bins for differential excitation values.
            If None, it is set to the maximum intensity value in the image.
        num_segments (int): Number of segments to divide the differential excitation range.
        num_bins_per_segment (int): Number of bins within each differential excitation segment.
        num_scales (int): Number of scales to compute the WLD descriptor.

    Returns:
        numpy.ndarray: 1D concatenated multi-scale WLD descriptor histogram.
    """
    # Create a Gaussian pyramid for the input image
    pyramid = [image.astype(np.float32)]  # Add the original image as the first level
    for _ in range(num_scales - 1):
        pyramid.append(cv2.pyrDown(pyramid[-1]))

    # Compute WLD descriptor at each scale and concatenate
    wld_descriptors = []
    for img in pyramid:
        #print('check',num_orientations)
        wld_descriptors.append(wld_descriptor(image, kernel_size=5, sigma=1.0))

    wld_multiscale = np.concatenate(wld_descriptors)
    return wld_multiscale

In [36]:
for i in range(10):
    print('Checking the Modified WLD Algorithm v2 10 folds\n')
    print('Iteration number',i)
    print('Train sampling beings..')
    dirpath='C:/Users/mahesh.inamdar/Desktop/codes/classes/run/train'
    folders=np.random.choice(os.listdir(dirpath),10)
    images=[]
    X_train=np.zeros(shape=(3000,72))
    y_train=[]
    a,no,c,N=0,0,0,0
    while N<3000:
        if N>3000:
            break
        if N<3000:
            N+=1
        n=np.random.choice(folders,1)
        y_train.append(get_label(n[0]))
        if 'AS' in n[0] and a<1000:
            img=np.random.choice(os.listdir(os.path.join(dirpath,n[0])),1)
            w=wld_descriptor_multiscale(preprocessing(resize_image(cv2.imread(os.path.join(dirpath,n[0],img[0]),0),(512,512))))
            X_train[N,0:len(w)]=w
            a+=1
        elif 'NS' in n[0] and no<1000:
            img=np.random.choice(os.listdir(os.path.join(dirpath,n[0])),1)
            w=wld_descriptor_multiscale(preprocessing(resize_image(cv2.imread(os.path.join(dirpath,n[0],img[0]),0),(512,512))))
            X_train[N,0:len(w)]=w
            no+=1
        elif 'CS' in n[0] and c<1000:
            img=np.random.choice(os.listdir(os.path.join(dirpath,n[0])),1)
            w=wld_descriptor_multiscale(preprocessing(resize_image(cv2.imread(os.path.join(dirpath,n[0],img[0]),0),(512,512))))
            X_train[N,0:len(w)]=w
            c+=1
    
    
    
    print('Test sampling beings..')
    dirpath='C:/Users/mahesh.inamdar/Desktop/codes/classes/run/test'
    folders=np.random.choice(os.listdir(dirpath),40)
    images=[]
    X_test=np.zeros(shape=(100,72))
    y_test=[]
    c=0
    while c<100:
        n=np.random.choice(folders,1)
        img=np.random.choice(os.listdir(os.path.join(dirpath,n[0])),1)
        w=wld_descriptor_multiscale(preprocessing(resize_image(cv2.imread(os.path.join(dirpath,n[0],img[0]),0),(512,512))))
        X_test[c,0:len(w)]=w
        y_test.append(get_label(n[0]))
        if c!=100:
            c+=1
        if c>100:
            break
    
    clf=LazyClassifier(verbose=0,ignore_warnings=True)
    models,predictions=clf.fit(X_train,X_test,y_train,y_test)
    print(models)

Checking the Modified WLD Algorithm v2 10 folds

Iteration number 0
Train sampling beings..
Test sampling beings..


 97%|███████████████████████████████████████████████████████████████████████████████▏  | 28/29 [00:07<00:00,  3.27it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001908 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16864
[LightGBM] [Info] Number of data points in the train set: 3000, number of used features: 72
[LightGBM] [Info] Start training from score -1.667773
[LightGBM] [Info] Start training from score -0.902223
[LightGBM] [Info] Start training from score -0.902223


100%|██████████████████████████████████████████████████████████████████████████████████| 29/29 [00:08<00:00,  3.35it/s]


                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
BernoulliNB                        0.50               0.66    None      0.44   
Perceptron                         0.53               0.63    None      0.49   
GaussianNB                         0.41               0.61    None      0.27   
NearestCentroid                    0.32               0.49    None      0.30   
PassiveAggressiveClassifier        0.45               0.46    None      0.42   
DecisionTreeClassifier             0.26               0.45    None      0.25   
QuadraticDiscriminantAnalysis      0.31               0.45    None      0.30   
XGBClassifier                      0.21               0.43    None      0.16   
LinearSVC                          0.26               0.42    None      0.20   
LogisticRegression                 0.26               0.42    None      0.20   
RidgeClassifierCV                  0.26 

 97%|███████████████████████████████████████████████████████████████████████████████▏  | 28/29 [00:07<00:00,  2.87it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001851 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16496
[LightGBM] [Info] Number of data points in the train set: 3000, number of used features: 72
[LightGBM] [Info] Start training from score -1.155183
[LightGBM] [Info] Start training from score -0.936493
[LightGBM] [Info] Start training from score -1.227583


100%|██████████████████████████████████████████████████████████████████████████████████| 29/29 [00:08<00:00,  3.42it/s]


                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
NearestCentroid                    0.44               0.53    None      0.43   
BernoulliNB                        0.36               0.49    None      0.29   
PassiveAggressiveClassifier        0.49               0.46    None      0.41   
GaussianNB                         0.42               0.45    None      0.38   
NuSVC                              0.38               0.43    None      0.38   
QuadraticDiscriminantAnalysis      0.33               0.42    None      0.32   
ExtraTreeClassifier                0.42               0.42    None      0.41   
KNeighborsClassifier               0.35               0.41    None      0.35   
RandomForestClassifier             0.29               0.41    None      0.27   
AdaBoostClassifier                 0.24               0.37    None      0.21   
LinearDiscriminantAnalysis         0.21 

IndexError: index 3000 is out of bounds for axis 0 with size 3000